# Задача 5: Интерполяция Лагранжа
---

### Задание

Рассчитать интерполяцию функции $\sin(x)$ на отрезке $[0, \pi]$ с использованием многочленов Лагранжа при вариации количества точек 3 и 5. Построить графики двух полученных многочленов Лагранжа и функции синуса на одном рисунке.

### Формула многочлена Лагранжа

Для $n$ точек $(x_0, y_0), (x_1, y_1), \ldots, (x_{n-1}, y_{n-1})$:

$$L(x) = \sum_{i=0}^{n-1} y_i \cdot l_i(x)$$

где базисные полиномы:

$$l_i(x) = \prod_{j=0, j \neq i}^{n-1} \frac{x - x_j}{x_i - x_j}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# Базисный полином Лагранжа l_i(x)
# l_i(x) = prod_{j≠i} (x - x_j) / (x_i - x_j)
# ============================================================

def lagrange_basis(x, x_points, i):
    """
    Вычисляет i-й базисный полином Лагранжа в точке x
    
    Args:
        x: точка, в которой вычисляется полином
        x_points: массив узлов интерполяции
        i: индекс базисного полинома
    """
    n = len(x_points)
    result = 1.0
    
    for j in range(n):
        if j != i:
            result *= (x - x_points[j]) / (x_points[i] - x_points[j])
    
    return result

In [ ]:
# ============================================================
# Интерполяционный многочлен Лагранжа
# L(x) = sum_{i=0}^{n-1} y_i * l_i(x)
# ============================================================

def lagrange_interpolation(x, x_points, y_points):
    """
    Вычисляет значение интерполяционного многочлена Лагранжа
    
    Args:
        x: точка или массив точек для вычисления
        x_points: узлы интерполяции
        y_points: значения функции в узлах
    """
    n = len(x_points)
    
    # Если x - массив, вычисляем для каждого элемента
    if isinstance(x, np.ndarray):
        return np.array([lagrange_interpolation(xi, x_points, y_points) for xi in x])
    
    # Для одной точки
    result = 0.0
    for i in range(n):
        result += y_points[i] * lagrange_basis(x, x_points, i)
    
    return result

---
## Интерполяция sin(x) на [0, π]

In [ ]:
# Функция для интерполяции
def f(x):
    return np.sin(x)

# Отрезок интерполяции
a, b = 0, np.pi

# Узлы интерполяции: 3 и 5 точек (равномерно распределенные)
n_points_list = [3, 5]

# Точки для построения графиков
x_plot = np.linspace(a, b, 200)
y_true = f(x_plot)

# Настройка графика
plt.figure(figsize=(12, 6))

# График исходной функции
plt.plot(x_plot, y_true, 'k-', linewidth=2, label='sin(x) (исходная функция)')

colors = ['blue', 'red']
markers = ['o', 's']

for idx, n_points in enumerate(n_points_list):
    # Равномерно распределенные узлы
    x_nodes = np.linspace(a, b, n_points)
    y_nodes = f(x_nodes)
    
    # Вычисление интерполяционного многочлена
    y_interp = lagrange_interpolation(x_plot, x_nodes, y_nodes)
    
    # График интерполяционного многочлена
    plt.plot(x_plot, y_interp, color=colors[idx], linestyle='--', 
             linewidth=1.5, label=f'Лагранж ({n_points} точек)')
    
    # Узлы интерполяции
    plt.scatter(x_nodes, y_nodes, color=colors[idx], marker=markers[idx], 
                s=100, zorder=5, edgecolors='black', linewidth=1)
    
    # Вычисление максимальной погрешности
    max_error = np.max(np.abs(y_true - y_interp))
    print(f"Максимальная погрешность ({n_points} точек): {max_error:.6f}")

plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Интерполяция sin(x) многочленами Лагранжа', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Анализ результатов

### Наблюдения:

1. **Точность интерполяции**: С увеличением количества узлов (от 3 до 5) погрешность интерполяции уменьшается

2. **Поведение на границах**: Многочлен Лагранжа точно проходит через все узлы интерполяции

3. **Гладкость**: Интерполяционный многочлен является гладкой функцией (бесконечно дифференцируемой)

### Свойства метода Лагранжа:

- ✓ Простота реализации
- ✓ Точное прохождение через узлы
- ✗ Вычислительная сложность O(n²) для каждой точки
- ✗ Возможны осцилляции при большом количестве узлов (эффект Рунге)

---
## Дополнительный тест: проверка базисных полиномов

Базисные полиномы должны удовлетворять свойству:

$$l_i(x_j) = \begin{cases} 1, & i = j \\ 0, & i \neq j \end{cases}$$

In [ ]:
# Проверка свойств базисных полиномов
x_test = np.array([0, 1, 2, 3])
n = len(x_test)

print("Проверка свойств базисных полиномов:")
print("l_i(x_j) должно быть 1 при i=j и 0 при i≠j\n")

for i in range(n):
    print(f"l_{i}(x_j):")
    for j in range(n):
        value = lagrange_basis(x_test[j], x_test, i)
        expected = 1.0 if i == j else 0.0
        status = "✓" if abs(value - expected) < 1e-10 else "✗"
        print(f"  j={j}: {value:.10f} (ожидается {expected:.1f}) {status}")
    print()